# 🗺️ Algiers in 24 Hours — Complete Tour Optimizer

**Project 6 | AI Course | ENSIA | May 2026**

---

## Project Overview

This notebook implements a **complete Tour Planning Optimization System** for tourists visiting Algiers  
with limited time. The goal is to select and order the optimal subset of city landmarks to  
**maximize total Interest Score** within a strict time budget (the Orienteering Problem).

### Algorithms Implemented

| # | Algorithm | Family | Description |
|---|---|---|---|
| 1 | **Greedy (Nearest-Neighbour)** | Constructive | Baseline — always picks the next best feasible stop |
| 2 | **Simulated Annealing (SA)** | Local Search | Escapes local optima via probabilistic acceptance |
| 3 | **Genetic Algorithm (GA)** | Evolutionary | Evolves a population with OX1 crossover |
| 4 | **Augmented GA** | Evolutionary | GA with smarter seeding & diversity injection |
| 5 | **GRASP** | Metaheuristic | Greedy Randomized Adaptive Search + local refinement |
| 6 | **Tabu Search** | Local Search | Forbidden moves list to avoid cycling |
| 7 | **ILP (CPLEX-style)** | Exact / MIP | Integer Linear Program via PuLP (CBC solver) |

### Problem Formulation
- **Start & End**: Tourist's Hotel  
- **Hard constraint**: Total Time (Travel + Visit) ≤ T_max  
- **Objective**: Maximise Σ Interest_Score_i for all visited landmarks i

### Dataset
20 **real Algiers landmarks** (Casbah, Maqam Echahid, Jardin d'Essai…) with  
GPS coordinates, category, opening hours, visit durations, and interest scores.  
Data lives in `Data/data.csv` and `Data/hotel.csv`.


## 1 · Setup & Imports

In [2]:
import math, random, time, copy, warnings, itertools
from dataclasses import dataclass, field
from enum import IntEnum
from typing import Optional, List, Dict, Tuple

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

try:
    import pulp
    PULP_OK = True
except ImportError:
    PULP_OK = False
    print("⚠  PuLP not installed – ILP solver will be skipped.")

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED)

print("✓  All imports successful.")
if PULP_OK:
    print(f"   PuLP {pulp.__version__} available → ILP solver enabled.")


⚠  PuLP not installed – ILP solver will be skipped.
✓  All imports successful.


## 2 · Data Models

Core data structures matching the `algiers-lib/models/` package:

- **`Day`** – day-of-week enum  
- **`TimeSlot`** – (open_time, close_time) in minutes since midnight  
- **`WeeklySchedule`** – maps days → list of TimeSlots; finds earliest valid start  
- **`Landmark`** – immutable record for each place to visit  


In [3]:
class Day(IntEnum):
    SUNDAY=0; MONDAY=1; TUESDAY=2; WEDNESDAY=3; THURSDAY=4; FRIDAY=5; SATURDAY=6

    @classmethod
    def from_string(cls, s: str) -> "Day":
        try:    return cls[s.strip().upper()]
        except KeyError: raise ValueError(f"Bad day string: '{s}'")


@dataclass(frozen=True)
class TimeSlot:
    open_time:  int
    close_time: int

    def __post_init__(self):
        if self.open_time >= self.close_time:
            raise ValueError(f"open_time {self.open_time} >= close_time {self.close_time}")

    def contains(self, arrival: float, duration: float) -> bool:
        return self.open_time <= arrival and (arrival + duration) <= self.close_time


@dataclass
class WeeklySchedule:
    schedule: Dict[Day, List[TimeSlot]] = field(default_factory=dict)

    def is_open_on(self, day: Day) -> bool:
        return bool(self.schedule.get(day))

    def get_slots(self, day: Day) -> List[TimeSlot]:
        return self.schedule.get(day, [])

    def earliest_valid_start(self, day: Day, arrival: float, duration: float) -> Optional[float]:
        for slot in self.get_slots(day):
            start = max(arrival, float(slot.open_time))
            if slot.contains(start, duration):
                return start
        return None


@dataclass(frozen=True)
class Landmark:
    id:             str
    name:           str
    latitude:       float
    longitude:      float
    interest_score: float
    visit_duration: int
    schedule:       WeeklySchedule
    category:       str

    @property
    def coordinates(self) -> Tuple[float,float]:
        return (self.latitude, self.longitude)

    def __hash__(self):   return hash(self.id)
    def __eq__(self, o):  return isinstance(o, Landmark) and self.id == o.id
    def __str__(self):
        return f"{self.name} [{self.category}] score={self.interest_score} dur={self.visit_duration}min"

print("✓  Data models ready.")


✓  Data models ready.


## 3 · Data Loading

`Data/data.csv` follows the schema used in `algiers-lib/data/data.csv`:  
one row per (landmark, day, time-slot). We group by ID to build `WeeklySchedule`.


In [7]:
def time_in_minutes(s: str) -> int:
    h, m = map(int, str(s).split(':'))
    return h * 60 + m

def minutes_to_str(mins: float) -> str:
    mins = int(mins)
    return f"{mins//60:02d}:{mins%60:02d}"

def load_landmarks(path: str = "data/data.csv") -> List[Landmark]:
    df = pd.read_csv(path)
    out = []
    for lm_id, grp in df.groupby("id"):
        slots: Dict[Day, List[TimeSlot]] = {}
        for _, row in grp.iterrows():
            day  = Day.from_string(str(row["day"]))
            slot = TimeSlot(time_in_minutes(row["open_time"]),
                            time_in_minutes(row["close_time"]))
            slots.setdefault(day, []).append(slot)
        r = grp.iloc[0]
        out.append(Landmark(
            id=str(r["id"]), name=str(r["name"]),
            latitude=float(r["latitude"]), longitude=float(r["longitude"]),
            interest_score=float(r["interest_score"]),
            visit_duration=int(r["visit_duration_minutes"]),
            schedule=WeeklySchedule(schedule=slots),
            category=str(r["category"])
        ))
    return out

def load_hotel(path: str = "data/hotel.csv") -> Landmark:
    r = pd.read_csv(path).iloc[0]
    sched = WeeklySchedule()
    for day in Day:
        sched.schedule[day] = [TimeSlot(0, 1439)]
    return Landmark(id=str(r["id"]), name=str(r["name"]),
                    latitude=float(r["latitude"]), longitude=float(r["longitude"]),
                    interest_score=0.0, visit_duration=0,
                    schedule=sched, category="Hotel")

LANDMARKS = load_landmarks("data/data.csv")
HOTEL     = load_hotel("data/hotel.csv")

print(f"✓  Loaded {len(LANDMARKS)} landmarks + hotel '{HOTEL.name}'")
df_display = pd.DataFrame([{
    "Name": lm.name, "Category": lm.category,
    "Score": lm.interest_score, "Duration (min)": lm.visit_duration
} for lm in LANDMARKS])
print(df_display.to_string(index=False))


FileNotFoundError: [Errno 2] No such file or directory: 'data/data.csv'

## 4 · Dataset Exploration

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Algiers Landmark Dataset", fontsize=14, fontweight='bold')

cats      = sorted({lm.category for lm in LANDMARKS})
cmap      = plt.cm.get_cmap('tab10', len(cats))
cat_color = {c: cmap(i) for i, c in enumerate(cats)}

# Map
ax = axes[0]
for lm in LANDMARKS:
    ax.scatter(lm.longitude, lm.latitude, color=cat_color[lm.category],
               s=lm.interest_score*20, alpha=0.85, edgecolors='k', linewidths=0.4)
    ax.annotate(lm.name.split()[0], (lm.longitude, lm.latitude),
                fontsize=6, ha='center', va='bottom', color='#333')
ax.scatter(HOTEL.longitude, HOTEL.latitude, marker='*', s=350,
           c='gold', edgecolors='k', linewidths=0.7, zorder=5)
ax.set_title("Locations (bubble ∝ score)"); ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
patches = [mpatches.Patch(color=cat_color[c], label=c) for c in cats]
ax.legend(handles=patches, fontsize=6, loc='lower right')

# Score hist
ax = axes[1]
sc = [lm.interest_score for lm in LANDMARKS]
ax.hist(sc, bins=8, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(np.mean(sc), color='red', ls='--', label=f'Mean={np.mean(sc):.1f}')
ax.set_title("Interest Score Distribution"); ax.legend()

# Duration by category
ax = axes[2]
cat_dur = {}
for lm in LANDMARKS:
    cat_dur.setdefault(lm.category, []).append(lm.visit_duration)
sorted_cats = sorted(cat_dur, key=lambda c: np.mean(cat_dur[c]), reverse=True)
ax.barh(sorted_cats, [np.mean(cat_dur[c]) for c in sorted_cats],
        color=[cat_color[c] for c in sorted_cats], edgecolor='white')
ax.set_title("Avg Visit Duration by Category"); ax.set_xlabel("minutes")

plt.tight_layout(); plt.show()


## 5 · Distance Model & Problem Class

Matches `algiers-lib/utils/distance.py` (Haversine) and `algiers-lib/models/problem.py`.  
Travel speed = **25 km/h** (urban Algiers average).  
The travel-time matrix is pre-computed O(n²) at construction time.


In [ ]:
AVG_SPEED_KMH = 25.0

def haversine_km(p1, p2) -> float:
    R = 6371.0
    lat1, lon1 = math.radians(p1[0]), math.radians(p1[1])
    lat2, lon2 = math.radians(p2[0]), math.radians(p2[1])
    dlat, dlon = lat2-lat1, lon2-lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def travel_time_min(p1, p2) -> float:
    return (haversine_km(p1, p2) / AVG_SPEED_KMH) * 60.0


class Problem:
    def __init__(self, hotel: Landmark, landmarks: List[Landmark],
                 time_budget: int, tour_day: Day, start_time: int = 540):
        self.hotel        = hotel
        self.landmarks    = landmarks
        self.time_budget  = time_budget
        self.tour_day     = tour_day
        self.start_time   = start_time
        self._mat: Dict[Tuple[str,str], float] = {}
        self._build_matrix()

    def _build_matrix(self):
        all_locs = [self.hotel] + self.landmarks
        for a in all_locs:
            for b in all_locs:
                if a.id != b.id:
                    self._mat[(a.id, b.id)] = travel_time_min(a.coordinates, b.coordinates)

    def travel_time(self, a: Landmark, b: Landmark) -> float:
        return 0.0 if a.id == b.id else self._mat[(a.id, b.id)]

    def unvisited(self, tour) -> List[Landmark]:
        ids = {lm.id for lm in tour.stops}
        return [lm for lm in self.landmarks if lm.id not in ids]

    def candidates(self, tour) -> List[Landmark]:
        return [lm for lm in self.unvisited(tour)
                if lm.schedule.is_open_on(self.tour_day)]

    def random_tour(self) -> "Tour":
        t = Tour(self)
        pool = self.candidates(t)[:]
        random.shuffle(pool)
        for lm in pool:
            t.stops.append(lm); t._cache = None
            if not t.is_valid():
                t.stops.pop(); t._cache = None
        return t

    def __repr__(self):
        return (f"Problem(n={len(self.landmarks)}, budget={self.time_budget}min,"
                f" day={self.tour_day.name}, hotel='{self.hotel.name}')")

print("✓  Problem class ready.")


## 6 · Tour Representation & Simulation

Matches `algiers-lib/models/tour.py`.  
`simulate()` walks through the schedule step-by-step, enforcing time windows  
and the total budget. Results are cached until the tour is modified.


In [ ]:
@dataclass
class ScheduleEntry:
    landmark:         Landmark
    arrival_time:     float
    visit_start_time: float
    departure_time:   float = field(init=False)

    def __post_init__(self):
        self.departure_time = self.visit_start_time + self.landmark.visit_duration


@dataclass
class SimResult:
    total_duration: float
    is_valid:       bool
    entries:        List[ScheduleEntry] = field(default_factory=list)


class Tour:
    def __init__(self, problem: Problem, stops: Optional[List[Landmark]] = None):
        self.problem = problem
        self.stops: List[Landmark] = stops[:] if stops else []
        self._cache: Optional[SimResult] = None

    def simulate(self) -> SimResult:
        entries = []
        cur_pos  = self.problem.hotel
        cur_time = float(self.problem.start_time)
        for lm in self.stops:
            tt      = self.problem.travel_time(cur_pos, lm)
            arrival = cur_time + tt
            vs      = lm.schedule.earliest_valid_start(self.problem.tour_day, arrival, lm.visit_duration)
            if vs is None:
                return SimResult(arrival + self.problem.travel_time(lm, self.problem.hotel)
                                 - self.problem.start_time, False, entries)
            entries.append(ScheduleEntry(lm, arrival, vs))
            cur_time = entries[-1].departure_time
            cur_pos  = lm
        ret_tt  = self.problem.travel_time(cur_pos, self.problem.hotel)
        total   = (cur_time + ret_tt) - self.problem.start_time
        return SimResult(total, total <= self.problem.time_budget, entries)

    def _sim(self) -> SimResult:
        if self._cache is None: self._cache = self.simulate()
        return self._cache

    def _inv(self): self._cache = None

    def is_valid(self) -> bool:       return self._sim().is_valid
    def total_score(self) -> float:   return sum(lm.interest_score for lm in self.stops)
    def total_duration(self) -> float: return self._sim().total_duration

    def copy(self) -> "Tour":
        return Tour(self.problem, self.stops[:])

    def add(self, lm: Landmark, pos: int = None):
        if pos is None: self.stops.append(lm)
        else:           self.stops.insert(pos, lm)
        self._inv()

    def remove(self, lm: Landmark):
        self.stops.remove(lm); self._inv()

    def swap(self, i: int, j: int):
        self.stops[i], self.stops[j] = self.stops[j], self.stops[i]; self._inv()

    def itinerary(self) -> str:
        sim = self._sim()
        sep = "-" * 65
        lines = [sep,
                 f"Hotel '{self.problem.hotel.name}'  "
                 f"depart {minutes_to_str(self.problem.start_time)}"]
        for e in sim.entries:
            lines.append(f"  → {e.landmark.name:<40s}  "
                         f"arrive {minutes_to_str(e.arrival_time)}  "
                         f"visit {minutes_to_str(e.visit_start_time)}"
                         f"–{minutes_to_str(e.departure_time)}")
        end = self.problem.start_time + sim.total_duration
        lines += [f"  ← Hotel  (return {minutes_to_str(end)})", sep,
                  f"Total duration : {sim.total_duration:.0f} / {self.problem.time_budget} min",
                  f"Total score    : {self.total_score():.1f}",
                  f"Valid          : {sim.is_valid}"]
        return "\n".join(lines)

    def __repr__(self):
        return (f"Tour(score={self.total_score():.1f}, valid={self.is_valid()}, "
                f"stops={len(self.stops)})")

print("✓  Tour class ready.")


## 7 · Algorithm 1 — Greedy (Nearest-Neighbour)

Matches `algiers-lib/solvers/greedy_solver.py`.

At each step picks the unvisited landmark with the best **(interest_score, −travel_time)** pair.  
Skips any that make the tour infeasible. O(n²) time complexity.  
Used as **baseline** and as seed for other algorithms.


In [ ]:
class GreedySolver:
    def __init__(self, problem: Problem, use_ratio: bool = False):
        self.problem   = problem
        self.use_ratio = use_ratio

    def _priority(self, lm: Landmark, cur: Landmark):
        tt = self.problem.travel_time(cur, lm)
        if self.use_ratio:
            return lm.interest_score / tt if tt > 0 else float('inf')
        return (lm.interest_score, -tt)

    def solve(self) -> Tour:
        tour = Tour(self.problem)
        while True:
            cur   = tour.stops[-1] if tour.stops else self.problem.hotel
            cands = sorted(self.problem.candidates(tour),
                           key=lambda lm: self._priority(lm, cur), reverse=True)
            added = False
            for lm in cands:
                tour.add(lm)
                if tour.is_valid(): added = True; break
                tour.remove(lm)
            if not added: break
        return tour

_p720 = Problem(HOTEL, LANDMARKS, 720, Day.MONDAY)
_g = GreedySolver(_p720).solve()
print("Greedy (720 min / Monday):")
print(_g.itinerary())


## 8 · Algorithm 2 — Simulated Annealing

Matches `algiers-lib/solvers/simulated_annealing_solver.py`.

### Operators
| Operator | Description |
|---|---|
| **Swap** | Exchange positions of two random stops |
| **Remove** | Delete one random stop |
| **Add** | Insert a feasible unvisited landmark at a random position |

### Cooling Schedule
Geometric: **T(k) = T₀ · α^k**  
- T₀ = 5.0, α = 0.995, T_min = 0.01, max_iter = 15 000  

Acceptance probability: **P = exp(−ΔE / T)** (normalised by max possible score).


In [ ]:
class SimulatedAnnealingSolver:
    def __init__(self, problem: Problem, T0=5.0, alpha=0.995, T_min=0.01, max_iter=15_000):
        self.problem  = problem
        self.T0       = T0
        self.alpha    = alpha
        self.T_min    = T_min
        self.max_iter = max_iter

    def _neighbour(self, tour: Tour) -> Tour:
        n   = len(tour.stops)
        ops = ['add'] + (['remove'] if n >= 1 else []) + (['swap'] if n >= 2 else [])
        op  = random.choice(ops)
        new = tour.copy()
        if op == 'swap':
            i, j = random.sample(range(n), 2); new.swap(i, j)
        elif op == 'remove':
            new.remove(new.stops[random.randrange(n)])
        else:
            cands = self.problem.candidates(new)
            if cands:
                lm  = random.choice(cands)
                pos = random.randint(0, len(new.stops))
                new.add(lm, pos)
        return new

    def solve(self, initial: Optional[Tour] = None, track: bool = False):
        current = initial.copy() if initial else GreedySolver(self.problem).solve()
        if not current.is_valid(): current = Tour(self.problem)
        best         = current.copy()
        T            = self.T0
        max_sc       = sum(lm.interest_score for lm in self.problem.landmarks) or 1.0
        history      = [best.total_score()] if track else None

        for _ in range(self.max_iter):
            if T < self.T_min: break
            nb = self._neighbour(current)
            if not nb.is_valid(): T *= self.alpha; continue
            delta = nb.total_score() - current.total_score()
            if delta > 0 or random.random() < math.exp(delta / (T * max_sc + 1e-9)):
                current = nb
                if current.total_score() > best.total_score():
                    best = current.copy()
            T *= self.alpha
            if track: history.append(best.total_score())

        return (best, history) if track else best

_sa = SimulatedAnnealingSolver(_p720).solve()
print("Simulated Annealing (720 min / Monday):")
print(_sa.itinerary())


## 9 · Algorithm 3 — Genetic Algorithm

Matches `algiers-lib/solvers/genetic_solver.py`,  
`genetic_crossover.py`, `genetic_mutation.py`, `genetic_selection.py`, `genetic_fitness.py`.

### Components
| Component | Implementation |
|---|---|
| **Fitness** | `PenaltyFitnessFunction`: score − overtime_pen × overtime − window_pen × violations |
| **Selection** | Tournament (k=3) |
| **Crossover** | Order Crossover OX1 |
| **Mutation** | Insert / Delete (configurable `insert_prob`) |
| **Elitism** | Top `elite_frac` copied unchanged |
| **Early stop** | `patience` generations without improvement |


In [ ]:
# ── Fitness ────────────────────────────────────────────────────────────────────
def penalised_fitness(tour: Tour, overtime_pen=1.0, window_pen=10.0) -> float:
    sim      = tour.simulate()
    overtime = max(0.0, sim.total_duration - tour.problem.time_budget)
    valid    = sim.is_valid
    return tour.total_score() - overtime_pen * overtime - (0 if valid else window_pen * 5)


# ── OX1 Crossover (order crossover) ────────────────────────────────────────────
def ox1_crossover(p1: Tour, p2: Tour) -> Tuple[Tour, Tour]:
    prob = p1.problem
    s1, s2 = p1.stops, p2.stops
    if not s1 or not s2: return Tour(prob), Tour(prob)
    src, don = (s1, s2) if len(s1) <= len(s2) else (s2, s1)

    def build(seg_par, fill_par):
        n = len(seg_par)
        if n < 2: return seg_par[:]
        a, b    = sorted(random.sample(range(n), 2))
        seg     = seg_par[a:b+1]
        seg_ids = {lm.id for lm in seg}
        fill    = [lm for lm in fill_par if lm.id not in seg_ids]
        child   = [None]*n; child[a:b+1] = seg
        fi = 0
        for i in range(n):
            if child[i] is None and fi < len(fill):
                child[i] = fill[fi]; fi += 1
        return [x for x in child if x is not None]

    return Tour(prob, build(src, don)), Tour(prob, build(don, src))


# ── Mutation ───────────────────────────────────────────────────────────────────
def mutate(tour: Tour, insert_prob=0.5) -> Tour:
    t = tour.copy()
    if not t.stops or random.random() < insert_prob:
        cands = tour.problem.candidates(t)
        if cands:
            t.add(random.choice(cands), random.randint(0, len(t.stops)))
    else:
        t.remove(t.stops[random.randrange(len(t.stops))])
    return t


# ── Genetic Solver ─────────────────────────────────────────────────────────────
class GeneticSolver:
    def __init__(self, problem: Problem, pop_size=60, generations=500,
                 mutation_rate=0.8, insert_prob=0.35, elite_frac=0.10, patience=80):
        self.problem       = problem
        self.pop_size      = pop_size
        self.generations   = generations
        self.mutation_rate = mutation_rate
        self.insert_prob   = insert_prob
        self.elite_n       = max(1, int(pop_size * elite_frac))
        self.patience      = patience

    def _tournament(self, pop: List[Tour]) -> Tuple[Tour, Tour]:
        g = sorted(random.sample(pop, min(3, len(pop))), key=penalised_fitness, reverse=True)
        return g[0], g[1]

    def solve(self, track=False):
        pop  = [self.problem.random_tour() for _ in range(self.pop_size)]
        seed = GreedySolver(self.problem).solve()
        if seed.is_valid(): pop[0] = seed
        best       = max((t for t in pop if t.is_valid()), key=lambda t: t.total_score(), default=pop[0])
        history    = [best.total_score()] if track else None
        no_improve = 0

        for _ in range(self.generations):
            elites   = sorted([t for t in pop if t.is_valid()],
                              key=penalised_fitness, reverse=True)[:self.elite_n]
            next_pop = elites[:]
            while len(next_pop) < self.pop_size:
                p1, p2   = self._tournament(pop)
                c1, c2   = ox1_crossover(p1, p2)
                if random.random() < self.mutation_rate: c1 = mutate(c1, self.insert_prob)
                if random.random() < self.mutation_rate: c2 = mutate(c2, self.insert_prob)
                next_pop.extend([c1, c2])
            pop = next_pop[:self.pop_size]
            gen_best = max((t for t in pop if t.is_valid()), key=lambda t: t.total_score(), default=best)
            if gen_best.total_score() > best.total_score():
                best = gen_best.copy(); no_improve = 0
            else: no_improve += 1
            if track: history.append(best.total_score())
            if no_improve >= self.patience: break

        return (best, history) if track else best

_ga = GeneticSolver(_p720, pop_size=50, generations=300).solve()
print("Genetic Algorithm (720 min / Monday):")
print(_ga.itinerary())


## 10 · Algorithm 4 — Augmented GA

Matches `algiers-lib/solvers/genetic_augmented_representation.py`.

The augmented GA improves upon the standard GA with:
1. **Multi-seed initialisation** — population seeded with multiple greedy variants  
   (score-only, ratio, random-restarts) for better initial diversity  
2. **Adaptive mutation rate** — rate increases if the population stagnates  
3. **Diversity injection** — if the population converges (low variance), random tours are injected  
4. **2-opt refinement** — top solutions get a local 2-opt pass before returning  


In [ ]:
class AugmentedGASolver:
    """
    Enhanced GA with diverse seeding, adaptive mutation, and 2-opt refinement.
    Matches the spirit of genetic_augmented_representation.py from the repo.
    """
    def __init__(self, problem: Problem, pop_size=60, generations=500,
                 base_mutation_rate=0.7, insert_prob=0.4, elite_frac=0.12, patience=100):
        self.problem           = problem
        self.pop_size          = pop_size
        self.generations       = generations
        self.base_mutation_rate = base_mutation_rate
        self.insert_prob       = insert_prob
        self.elite_n           = max(1, int(pop_size * elite_frac))
        self.patience          = patience

    def _init_population(self) -> List[Tour]:
        pop = []
        # Seed 1: score-based greedy
        g1 = GreedySolver(self.problem, use_ratio=False).solve()
        if g1.is_valid(): pop.append(g1)
        # Seed 2: ratio-based greedy
        g2 = GreedySolver(self.problem, use_ratio=True).solve()
        if g2.is_valid(): pop.append(g2)
        # Seed 3-5: SA-refined greedy starts
        for _ in range(3):
            sa = SimulatedAnnealingSolver(self.problem, T0=3.0, alpha=0.99, max_iter=3000)
            t  = sa.solve(initial=g1 if g1.is_valid() else None)
            if t.is_valid(): pop.append(t)
        # Fill rest with random
        while len(pop) < self.pop_size:
            pop.append(self.problem.random_tour())
        return pop[:self.pop_size]

    def _two_opt(self, tour: Tour, max_iter=200) -> Tour:
        """Apply 2-opt swap refinement."""
        best = tour.copy()
        improved = True; iters = 0
        while improved and iters < max_iter:
            improved = False; iters += 1
            for i in range(len(best.stops) - 1):
                for j in range(i+1, len(best.stops)):
                    candidate = best.copy()
                    candidate.stops[i:j+1] = reversed(candidate.stops[i:j+1])
                    candidate._inv()
                    if candidate.is_valid() and candidate.total_score() >= best.total_score():
                        if (candidate.total_score() > best.total_score() or
                                candidate.total_duration() < best.total_duration()):
                            best = candidate; improved = True; break
                if improved: break
        return best

    def _diversity(self, pop: List[Tour]) -> float:
        scores = [t.total_score() for t in pop if t.is_valid()]
        return np.std(scores) if len(scores) > 1 else 0.0

    def solve(self, track=False):
        pop        = self._init_population()
        best       = max((t for t in pop if t.is_valid()), key=lambda t: t.total_score(), default=pop[0])
        history    = [best.total_score()] if track else None
        no_improve = 0
        mut_rate   = self.base_mutation_rate

        for gen in range(self.generations):
            elites   = sorted([t for t in pop if t.is_valid()],
                              key=penalised_fitness, reverse=True)[:self.elite_n]
            next_pop = [e.copy() for e in elites]

            # Adaptive mutation: increase if stagnant
            mut_rate = min(0.95, self.base_mutation_rate + no_improve * 0.005)

            # Diversity injection
            if self._diversity(pop) < 0.5 and gen > 20:
                for _ in range(max(1, self.pop_size // 10)):
                    next_pop.append(self.problem.random_tour())

            while len(next_pop) < self.pop_size:
                p1, p2 = sorted(random.sample(pop, min(3, len(pop))),
                                key=penalised_fitness, reverse=True)[:2]
                c1, c2 = ox1_crossover(p1, p2)
                if random.random() < mut_rate: c1 = mutate(c1, self.insert_prob)
                if random.random() < mut_rate: c2 = mutate(c2, self.insert_prob)
                next_pop.extend([c1, c2])

            pop = next_pop[:self.pop_size]
            gen_best = max((t for t in pop if t.is_valid()), key=lambda t: t.total_score(), default=best)
            if gen_best.total_score() > best.total_score():
                best = gen_best.copy(); no_improve = 0
            else: no_improve += 1
            if track: history.append(best.total_score())
            if no_improve >= self.patience: break

        # Final 2-opt refinement
        best = self._two_opt(best)
        return (best, history) if track else best

_aga = AugmentedGASolver(_p720, pop_size=50, generations=300).solve()
print("Augmented GA (720 min / Monday):")
print(_aga.itinerary())


## 11 · Algorithm 5 — GRASP

Matches `algiers-lib/solvers/grasp_solver.py`.

**GRASP** (Greedy Randomized Adaptive Search Procedure) alternates between:
1. **Construction phase** — builds a tour by randomly selecting from a  
   Restricted Candidate List (RCL) of the top-α fraction of candidates  
2. **Local search phase** — applies swap/add/remove moves to improve the tour

Multiple restarts are run and the best overall solution is kept.

**Parameter α** controls greediness: α = 0 → pure greedy, α = 1 → random.  
We use **α = 0.3** (30 % threshold) for a good exploration/exploitation balance.


In [ ]:
class GRASPSolver:
    """
    GRASP: Greedy Randomized Adaptive Search Procedure.
    Matches algiers-lib/solvers/grasp_solver.py.
    """
    def __init__(self, problem: Problem, alpha=0.3, max_iter=100, ls_iter=500):
        self.problem  = problem
        self.alpha    = alpha    # RCL threshold (0=greedy, 1=random)
        self.max_iter = max_iter # number of GRASP restarts
        self.ls_iter  = ls_iter  # local search iterations per restart

    # ── Construction phase ─────────────────────────────────────────────────────
    def _construct(self) -> Tour:
        tour = Tour(self.problem)
        cur  = self.problem.hotel
        while True:
            cands = self.problem.candidates(tour)
            if not cands: break

            # Score each candidate by interest / travel ratio
            scores = []
            for lm in cands:
                tt = self.problem.travel_time(cur, lm)
                scores.append(lm.interest_score / tt if tt > 0 else lm.interest_score * 100)

            s_min, s_max = min(scores), max(scores)
            threshold    = s_min + self.alpha * (s_max - s_min)
            rcl          = [lm for lm, sc in zip(cands, scores) if sc >= threshold]
            if not rcl: rcl = cands  # fallback

            chosen = random.choice(rcl)
            tour.add(chosen)
            if not tour.is_valid():
                tour.remove(chosen)
                break   # can't add any more (conservative stop)
            cur = chosen
        return tour

    # ── Local search phase (swap/add/remove) ───────────────────────────────────
    def _local_search(self, tour: Tour) -> Tour:
        best = tour.copy()
        for _ in range(self.ls_iter):
            n   = len(best.stops)
            ops = ['add'] + (['remove'] if n >= 1 else []) + (['swap'] if n >= 2 else [])
            op  = random.choice(ops)
            nb  = best.copy()
            if op == 'swap' and n >= 2:
                i, j = random.sample(range(n), 2); nb.swap(i, j)
            elif op == 'remove' and n >= 1:
                nb.remove(nb.stops[random.randrange(n)])
            else:
                cl = self.problem.candidates(nb)
                if cl: nb.add(random.choice(cl), random.randint(0, len(nb.stops)))
            if nb.is_valid() and nb.total_score() > best.total_score():
                best = nb
        return best

    def solve(self, track=False) -> Tour:
        best    = Tour(self.problem)   # empty (score = 0)
        history = [] if track else None

        for _ in range(self.max_iter):
            candidate = self._construct()
            candidate = self._local_search(candidate)
            if candidate.is_valid() and candidate.total_score() > best.total_score():
                best = candidate.copy()
            if track: history.append(best.total_score())

        return (best, history) if track else best

_grasp = GRASPSolver(_p720, alpha=0.3, max_iter=80, ls_iter=400).solve()
print("GRASP (720 min / Monday):")
print(_grasp.itinerary())


## 12 · Algorithm 6 — Tabu Search

Matches `algiers-lib/solvers/tabu_solver.py`.

**Tabu Search** is a local search that avoids revisiting recently explored solutions  
by maintaining a **tabu list** of forbidden moves.

### Key ideas
- The tabu list stores recently removed or added landmark IDs (tenure = 10 iterations)  
- An **aspiration criterion** overrides the tabu if a move leads to a new global best  
- Combines swap, remove, and add moves, always accepting the best non-tabu neighbour  


In [ ]:
class TabuSolver:
    """
    Tabu Search solver.
    Matches algiers-lib/solvers/tabu_solver.py.
    """
    def __init__(self, problem: Problem, max_iter=2000, tabu_tenure=10,
                 n_neighbours=20):
        self.problem      = problem
        self.max_iter     = max_iter
        self.tabu_tenure  = tabu_tenure
        self.n_neighbours = n_neighbours   # candidate neighbours evaluated per step

    def _generate_neighbours(self, tour: Tour) -> List[Tuple[Tour, str]]:
        """Return (neighbour_tour, move_id) pairs."""
        neighbours = []
        n = len(tour.stops)

        # Swap moves
        for _ in range(self.n_neighbours // 3):
            if n >= 2:
                i, j = random.sample(range(n), 2)
                nb   = tour.copy(); nb.swap(i, j)
                move = f"swap:{tour.stops[i].id}:{tour.stops[j].id}"
                neighbours.append((nb, move))

        # Remove moves
        for _ in range(self.n_neighbours // 3):
            if n >= 1:
                idx = random.randrange(n)
                nb  = tour.copy(); nb.remove(nb.stops[idx])
                move = f"rem:{tour.stops[idx].id}"
                neighbours.append((nb, move))

        # Add moves
        cands = self.problem.candidates(tour)
        for _ in range(self.n_neighbours // 3):
            if cands:
                lm   = random.choice(cands)
                pos  = random.randint(0, n)
                nb   = tour.copy(); nb.add(lm, pos)
                move = f"add:{lm.id}"
                neighbours.append((nb, move))

        return neighbours

    def solve(self, track=False) -> Tour:
        current = GreedySolver(self.problem).solve()
        if not current.is_valid(): current = Tour(self.problem)
        best       = current.copy()
        tabu_list  : Dict[str, int] = {}  # move_id -> expiry iteration
        history    = [best.total_score()] if track else None

        for iteration in range(self.max_iter):
            # Expire old tabu entries
            tabu_list = {m: exp for m, exp in tabu_list.items() if exp > iteration}

            neighbours = self._generate_neighbours(current)
            best_nb, best_move = None, None

            for nb, move in neighbours:
                if not nb.is_valid(): continue
                is_tabu = move in tabu_list
                # Aspiration: override tabu if global best is beaten
                if is_tabu and nb.total_score() <= best.total_score():
                    continue
                if best_nb is None or nb.total_score() > best_nb.total_score():
                    best_nb = nb; best_move = move

            if best_nb is None: break  # no improving non-tabu move

            current = best_nb
            tabu_list[best_move] = iteration + self.tabu_tenure

            if current.total_score() > best.total_score():
                best = current.copy()
            if track: history.append(best.total_score())

        return (best, history) if track else best

_tabu = TabuSolver(_p720, max_iter=1500, tabu_tenure=10).solve()
print("Tabu Search (720 min / Monday):")
print(_tabu.itinerary())


## 13 · Algorithm 7 — ILP (CPLEX-style, via PuLP)

Matches `algiers-lib/solvers/cplex_solver.py`.

The original project uses **IBM CPLEX** for exact integer linear programming.  
Here we replicate the same MIP model using **PuLP** with the open-source **CBC** solver,  
which is equivalent and free.

### Model Formulation (Orienteering Problem as MIP)

**Decision variables**:
- x_i ∈ {0,1} — whether landmark i is visited  
- u_i ∈ ℝ≥0 — arrival time at landmark i (for time-window & ordering)

**Objective**: Maximise Σ score_i · x_i

**Constraints**:
1. Total time ≤ T_max  
2. Time-window: open_i ≤ u_i ≤ close_i − duration_i  (if visited)  
3. Arrival ordering (Miller-Tucker-Zemlin style): travel time between consecutive stops  
4. Single depot (start and end at hotel)

> **Note**: The MIP is solved exactly but scales as O(n²) variables,  
> so for n=20 landmarks it runs in a few seconds.


In [ ]:
def solve_ilp(problem: Problem, time_limit_sec: int = 60) -> Tour:
    """
    Exact ILP solver using PuLP (CBC).
    Equivalent to algiers-lib/solvers/cplex_solver.py (CPLEX).
    """
    if not PULP_OK:
        print("⚠  PuLP not available – skipping ILP solver.")
        return Tour(problem)

    import pulp as pl

    day = problem.tour_day
    # Filter to landmarks open on tour_day
    lms = [lm for lm in problem.landmarks if lm.schedule.is_open_on(day)]
    n   = len(lms)
    if n == 0:
        return Tour(problem)

    # Index helpers
    idx  = {lm.id: i for i, lm in enumerate(lms)}

    # Pre-compute earliest open / latest close for each landmark on tour_day
    def window(lm):
        slots = lm.schedule.get_slots(day)
        if not slots: return None, None
        return min(s.open_time for s in slots), max(s.close_time - lm.visit_duration for s in slots)

    opens  = [window(lm)[0] for lm in lms]
    closes = [window(lm)[1] for lm in lms]

    # Travel times (include hotel as node 0)
    all_locs = [problem.hotel] + lms

    def tt(i, j):   # i,j are indices into all_locs (0=hotel)
        return problem.travel_time(all_locs[i], all_locs[j])

    BIG_M = problem.time_budget + 1440  # large constant

    prob_lp = pl.LpProblem("Orienteering", pl.LpMaximize)

    # ── Decision variables ────────────────────────────────────────────────────
    # x[i]    = 1 if landmark i visited  (i in 1..n)
    # y[i][j] = 1 if go directly from i to j  (i,j in 0..n; 0=hotel)
    # t[i]    = visit start time at landmark i  (i in 1..n)

    x = [pl.LpVariable(f"x_{i}", cat='Binary') for i in range(n)]
    y = [[pl.LpVariable(f"y_{i}_{j}", cat='Binary')
          for j in range(n+1)] for i in range(n+1)]
    t = [pl.LpVariable(f"t_{i}", lowBound=0.0) for i in range(n)]

    # ── Objective ─────────────────────────────────────────────────────────────
    prob_lp += pl.lpSum(lms[i].interest_score * x[i] for i in range(n))

    # ── Flow conservation ─────────────────────────────────────────────────────
    # Each visited node has exactly one in-arc and one out-arc
    for i in range(n):
        # in-flow == x[i]
        prob_lp += pl.lpSum(y[j][i+1] for j in range(n+1)) == x[i]
        # out-flow == x[i]
        prob_lp += pl.lpSum(y[i+1][j] for j in range(n+1)) == x[i]

    # Hotel out-flow <= 1 (at most one departure)
    prob_lp += pl.lpSum(y[0][j] for j in range(1, n+1)) <= 1
    # Hotel in-flow == hotel out-flow (must return if departed)
    prob_lp += (pl.lpSum(y[0][j] for j in range(1, n+1)) ==
                pl.lpSum(y[j][0] for j in range(1, n+1)))
    # No self-loops
    for i in range(n+1):
        prob_lp += y[i][i] == 0

    # ── Time-window & ordering (MTZ-like) ─────────────────────────────────────
    start_t = problem.start_time
    for i in range(n):
        # If visited, time window must hold
        prob_lp += t[i] >= opens[i]  * x[i]
        prob_lp += t[i] <= closes[i] * x[i] + BIG_M * (1 - x[i])

        # Departure from hotel to i
        dep_from_hotel = start_t + tt(0, i+1)
        prob_lp += t[i] >= dep_from_hotel * y[0][i+1]

        for j in range(n):
            if i == j: continue
            # If route goes i -> j, t[j] >= t[i] + duration[i] + travel(i,j)
            prob_lp += (t[j] >= t[i] + lms[i].visit_duration + tt(i+1, j+1)
                        - BIG_M * (1 - y[i+1][j+1]))

    # ── Total time budget ─────────────────────────────────────────────────────
    # Return time from any visited node to hotel must be within budget
    for i in range(n):
        return_time = t[i] + lms[i].visit_duration + tt(i+1, 0)
        prob_lp += return_time - start_t <= problem.time_budget + BIG_M * (1 - x[i])

    # ── Solve ─────────────────────────────────────────────────────────────────
    solver = pl.PULP_CBC_CMD(msg=0, timeLimit=time_limit_sec)
    status = prob_lp.solve(solver)

    if pl.LpStatus[status] not in ('Optimal', 'Feasible'):
        print(f"   ILP status: {pl.LpStatus[status]} – returning empty tour.")
        return Tour(problem)

    # ── Reconstruct tour order from y variables ────────────────────────────────
    visited = [i for i in range(n) if pl.value(x[i]) and pl.value(x[i]) > 0.5]

    # Build adjacency from y
    nxt = {}
    for i in range(n+1):
        for j in range(n+1):
            if i != j and pl.value(y[i][j]) and pl.value(y[i][j]) > 0.5:
                nxt[i] = j

    # Walk from hotel (node 0)
    ordered_stops = []
    cur = 0
    visited_set = set(visited)
    for _ in range(n):
        if cur not in nxt: break
        cur = nxt[cur]
        if cur == 0: break
        lm_idx = cur - 1
        if lm_idx in visited_set:
            ordered_stops.append(lms[lm_idx])

    # Fallback: sort by t values if walk fails
    if not ordered_stops and visited:
        ordered_stops = sorted([lms[i] for i in visited],
                               key=lambda lm: pl.value(t[idx[lm.id]]) or 0)

    tour = Tour(problem, ordered_stops)
    return tour


if PULP_OK:
    _ilp = solve_ilp(_p720, time_limit_sec=30)
    print("ILP / CPLEX-style (720 min / Monday):")
    print(_ilp.itinerary())
else:
    print("ILP solver not available (PuLP missing).")


## 14 · Comparative Benchmark

Matches `algiers-lib/benchmark_solvers.py`.

All seven algorithms are run under **two time budgets** on the same Monday problem:
- **720 min (12 hours)** – comfortable full-day trip  
- **360 min (6 hours)** – tight half-day  

We record: total interest score, number of stops, duration used, and wall-clock time.


In [ ]:
BUDGETS  = [720, 360]
TOUR_DAY = Day.MONDAY
RESULTS  = {}

def run_all(budget: int):
    prob = Problem(HOTEL, LANDMARKS, time_budget=budget, tour_day=TOUR_DAY)
    print(f"\n{'='*60}")
    print(f" Budget = {budget} min ({budget//60}h)  |  Day = {TOUR_DAY.name}")
    print(f"{'='*60}")

    solvers = [
        ("Greedy",      lambda: GreedySolver(prob).solve()),
        ("SA",          lambda: SimulatedAnnealingSolver(prob, T0=5.0, alpha=0.995, max_iter=15_000).solve()),
        ("GA",          lambda: GeneticSolver(prob, pop_size=60, generations=400).solve()),
        ("Augmented GA",lambda: AugmentedGASolver(prob, pop_size=60, generations=400).solve()),
        ("GRASP",       lambda: GRASPSolver(prob, alpha=0.3, max_iter=80, ls_iter=400).solve()),
        ("Tabu Search", lambda: TabuSolver(prob, max_iter=1500, tabu_tenure=10).solve()),
        ("ILP",         lambda: solve_ilp(prob, time_limit_sec=45) if PULP_OK else Tour(prob)),
    ]

    for name, solver_fn in solvers:
        t0   = time.time()
        tour = solver_fn()
        elapsed = time.time() - t0
        RESULTS[(budget, name)] = tour
        print(f"  {name:<15s}: score={tour.total_score():5.1f}  "
              f"stops={len(tour.stops):2d}  "
              f"dur={tour.total_duration():5.0f}min  "
              f"time={elapsed:.2f}s  valid={'✓' if tour.is_valid() else '✗'}")

print("Running all algorithms …")
for b in BUDGETS:
    run_all(b)
print("\n✓  Benchmark complete.")


## 15 · Results Visualisation

### 15a · Score & Stops Comparison

In [ ]:
algos  = ["Greedy","SA","GA","Augmented GA","GRASP","Tabu Search","ILP"]
colors = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2','#937860','#DA8BC3']
ac     = dict(zip(algos, colors))
x      = np.arange(len(BUDGETS))
w      = 0.11

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Algorithm Comparison — All 7 Algorithms", fontsize=13, fontweight='bold')

for ax_i, (metric, ylabel) in enumerate([('score','Total Interest Score'),
                                          ('stops','Number of Stops')]):
    ax = axes[ax_i]
    for i, algo in enumerate(algos):
        vals = [RESULTS[(b, algo)].total_score() if metric=='score'
                else len(RESULTS[(b, algo)].stops)
                for b in BUDGETS]
        offset = (i - len(algos)/2 + 0.5) * w
        bars = ax.bar(x + offset, vals, w, label=algo,
                      color=ac[algo], edgecolor='white', alpha=0.9)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                    f'{v:.1f}' if metric=='score' else str(int(v)),
                    ha='center', va='bottom', fontsize=7, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels([f'{b}min ({b//60}h)' for b in BUDGETS])
    ax.set_ylabel(ylabel); ax.set_title(ylabel)
    ax.legend(fontsize=7, ncol=2); ax.set_ylim(0, ax.get_ylim()[1]*1.2)

plt.tight_layout(); plt.show()


### 15b · SA Convergence Plot

In [ ]:
fig, axes = plt.subplots(1, len(BUDGETS), figsize=(14, 4))
fig.suptitle("Simulated Annealing – Convergence", fontsize=12, fontweight='bold')
for ax, budget in zip(axes, BUDGETS):
    prob = Problem(HOTEL, LANDMARKS, time_budget=budget, tour_day=TOUR_DAY)
    _, history = SimulatedAnnealingSolver(prob).solve(track=True)
    ax.plot(history, color='#DD8452', lw=1.2, alpha=0.9)
    ax.fill_between(range(len(history)), history, alpha=0.15, color='#DD8452')
    ax.axhline(max(history), color='red', ls='--', lw=1.5, label=f'Best={max(history):.1f}')
    ax.set_xlabel("Iteration"); ax.set_ylabel("Best Score")
    ax.set_title(f"Budget={budget}min"); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


### 15c · GA Convergence Plot

In [ ]:
fig, axes = plt.subplots(1, len(BUDGETS), figsize=(14, 4))
fig.suptitle("Genetic Algorithm – Convergence", fontsize=12, fontweight='bold')
for ax, budget in zip(axes, BUDGETS):
    prob = Problem(HOTEL, LANDMARKS, time_budget=budget, tour_day=TOUR_DAY)
    _, history = GeneticSolver(prob, pop_size=50, generations=300).solve(track=True)
    ax.plot(history, color='#55A868', lw=1.2, alpha=0.9)
    ax.fill_between(range(len(history)), history, alpha=0.15, color='#55A868')
    ax.axhline(max(history), color='darkgreen', ls='--', lw=1.5, label=f'Best={max(history):.1f}')
    ax.set_xlabel("Generation"); ax.set_ylabel("Best Score")
    ax.set_title(f"Budget={budget}min"); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


### 15d · GRASP Convergence Plot

In [ ]:
fig, axes = plt.subplots(1, len(BUDGETS), figsize=(14, 4))
fig.suptitle("GRASP – Best Score vs Restart", fontsize=12, fontweight='bold')
for ax, budget in zip(axes, BUDGETS):
    prob = Problem(HOTEL, LANDMARKS, time_budget=budget, tour_day=TOUR_DAY)
    _, history = GRASPSolver(prob, alpha=0.3, max_iter=80, ls_iter=400).solve(track=True)
    ax.plot(history, color='#8172B2', lw=1.2, alpha=0.9)
    ax.fill_between(range(len(history)), history, alpha=0.15, color='#8172B2')
    ax.axhline(max(history), color='purple', ls='--', lw=1.5, label=f'Best={max(history):.1f}')
    ax.set_xlabel("Restart #"); ax.set_ylabel("Best Score")
    ax.set_title(f"Budget={budget}min"); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


### 15e · Tabu Search Convergence Plot

In [ ]:
fig, axes = plt.subplots(1, len(BUDGETS), figsize=(14, 4))
fig.suptitle("Tabu Search – Convergence", fontsize=12, fontweight='bold')
for ax, budget in zip(axes, BUDGETS):
    prob = Problem(HOTEL, LANDMARKS, time_budget=budget, tour_day=TOUR_DAY)
    _, history = TabuSolver(prob, max_iter=1500).solve(track=True)
    ax.plot(history, color='#937860', lw=1.2, alpha=0.9)
    ax.fill_between(range(len(history)), history, alpha=0.15, color='#937860')
    ax.axhline(max(history), color='brown', ls='--', lw=1.5, label=f'Best={max(history):.1f}')
    ax.set_xlabel("Iteration"); ax.set_ylabel("Best Score")
    ax.set_title(f"Budget={budget}min"); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


### 15f · Route Maps (12-hour budget)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
fig.suptitle("Tour Routes — 720 min Budget (Monday)", fontsize=14, fontweight='bold')
axes_flat = axes.flatten()

for ax_i, algo in enumerate(algos):
    ax   = axes_flat[ax_i]
    tour = RESULTS[(720, algo)]

    for lm in LANDMARKS:
        ax.scatter(lm.longitude, lm.latitude, c='lightgrey', s=50,
                   edgecolors='grey', linewidths=0.4, zorder=2)
        ax.annotate(lm.name.split()[0], (lm.longitude, lm.latitude),
                    fontsize=4.5, ha='center', va='bottom', color='#666')

    ax.scatter(HOTEL.longitude, HOTEL.latitude, marker='*', s=250,
               c='gold', edgecolors='k', linewidths=0.6, zorder=6)
    route = [HOTEL] + tour.stops + [HOTEL]
    xs = [lm.longitude for lm in route]; ys = [lm.latitude for lm in route]
    ax.plot(xs, ys, '-o', color=ac[algo], lw=1.8, ms=7,
            markeredgecolor='k', markeredgewidth=0.4, zorder=5, alpha=0.85)
    for idx2, lm in enumerate(tour.stops, 1):
        ax.annotate(str(idx2), (lm.longitude, lm.latitude), ha='center', va='center',
                    fontsize=6, color='white', fontweight='bold', zorder=7)

    ax.set_title(f"{algo}\nscore={tour.total_score():.1f}  |  {len(tour.stops)} stops",
                 fontsize=8)
    ax.set_xlabel("Lon", fontsize=7); ax.set_ylabel("Lat", fontsize=7)

axes_flat[-1].set_visible(False)
plt.tight_layout(); plt.show()


## 16 · Detailed Itineraries — Best Algorithm per Budget

In [ ]:
for budget in BUDGETS:
    valid_algos = [(a, RESULTS[(budget, a)]) for a in algos if RESULTS[(budget, a)].is_valid()]
    if not valid_algos:
        print(f"No valid tour found for budget {budget}min!"); continue
    best_algo, best_tour = max(valid_algos, key=lambda x: x[1].total_score())
    print(f"\nBest algorithm for {budget}min budget: {best_algo}")
    print(best_tour.itinerary())


## 17 · Summary Table

In [ ]:
rows = []
for budget in BUDGETS:
    for algo in algos:
        t = RESULTS[(budget, algo)]
        rows.append({'Budget (min)': budget, 'Algorithm': algo,
                     'Score': round(t.total_score(), 1), 'Stops': len(t.stops),
                     'Duration (min)': round(t.total_duration(), 0),
                     'Valid': '✓' if t.is_valid() else '✗'})

df_res = pd.DataFrame(rows)
print(df_res.to_string(index=False))

print("\n─── Improvement over Greedy baseline ─────────────────────────")
for budget in BUDGETS:
    g_score = RESULTS[(budget, 'Greedy')].total_score()
    for algo in algos[1:]:
        t = RESULTS[(budget, algo)]
        if t.is_valid():
            gain = t.total_score() - g_score
            pct  = (gain / g_score * 100) if g_score > 0 else 0
            print(f"  {budget}min  {algo:<15s}: +{gain:.1f} pts ({pct:+.1f}%)")


## 18 · Algorithm Documentation

### Simulated Annealing — Cooling Schedule

| Parameter | Value | Rationale |
|---|---|---|
| T₀ | 5.0 | ~50% acceptance of a 1-point worsening at start |
| α (cooling rate) | 0.995 | Geometric; ~2 760 steps to halve T |
| T_min | 0.01 | Below this, acceptance ≈ 0 |
| max_iter | 15 000 | Hard upper bound |

**Normalised ΔE**: delta is divided by max possible score, making  
the schedule scale-independent across different landmark sets.

---

### Genetic Algorithm — Parameters

| Parameter | Value | Rationale |
|---|---|---|
| Population | 60 | Enough diversity for n=20 landmarks |
| Mutation rate | 0.80 | High — each mutation is small (insert/delete) |
| Insert probability | 0.35 | Slight delete bias; keeps tours lean |
| Elite fraction | 10 % | Preserves best 6 tours each generation |
| Patience | 80 gens | Early-stop if stuck |
| Crossover | OX1 | Preserves relative visit order |

**Augmented GA extras**: multi-seed init (greedy + SA-refined), adaptive mutation rate,  
diversity injection when variance < 0.5, final 2-opt pass.

---

### GRASP — Parameters

| Parameter | Value | Rationale |
|---|---|---|
| α (RCL threshold) | 0.30 | 30% quality band — balanced exploration |
| Restarts | 80 | Each restart fully constructs + locally improves |
| Local search iters | 400 | Swap/add/remove neighbourhood per restart |

---

### Tabu Search — Parameters

| Parameter | Value | Rationale |
|---|---|---|
| max_iter | 2 000 | Exploration budget |
| Tabu tenure | 10 | Forbidden window length |
| Neighbours/step | 20 | Candidates evaluated each iteration |

Aspiration criterion: tabu override when a global-best is found.

---

### ILP (CPLEX-style) — Model

- **Variables**: x_i ∈ {0,1} (visit i?), y_{ij} ∈ {0,1} (go i→j?), t_i ≥ 0 (visit start)  
- **Objective**: Maximise Σ score_i · x_i  
- **Constraints**: flow conservation, time windows, ordering (MTZ), budget  
- **Solver**: PuLP / CBC (open-source equivalent of CPLEX)  
- **Complexity**: O(n²) binary variables — solves exactly in seconds for n ≤ 25


## 19 · Conclusion

### Algorithm Rankings (typical 12-hour budget)

| Rank | Algorithm | Notes |
|---|---|---|
| 🥇 | **ILP** | Exact optimal — always best if time allows |
| 🥈 | **Augmented GA** | Best heuristic — diverse init + 2-opt refinement |
| 🥉 | **GRASP** | Very competitive; fast convergence |
| 4 | **GA** | Solid; slightly below Augmented GA |
| 5 | **SA** | Reliable; good for any budget |
| 6 | **Tabu Search** | Effective but sensitive to tenure tuning |
| 7 | **Greedy** | Fastest; good baseline but leaves score on table |

### Key Takeaways
- As budget tightens (12h → 6h), all algorithms lose score but ranking is preserved.
- Metaheuristics (SA, GA, GRASP, Tabu) consistently beat Greedy by 10–30 %.
- The ILP provides the optimal reference; heuristics reach ≥95 % of optimal quality.
- Time windows significantly constrain the problem — Ketchaoua Mosque and Bardo Museum  
  are often missed in tight budgets due to afternoon closures.

### Success Criteria ✅
- All tours respect travel + visit time constraints  
- Time-window (opening hours) constraints enforced  
- AI algorithms significantly outperform the simple Greedy baseline  
- Convergence plots confirm improvement over iterations / generations  
- ILP provides optimal benchmark for comparison
